# Repaso paso a paso: programación por restricciones con OR-Tools
**Tópicos en Ciencias de la Computación · Guía práctica para Google Colab**

**Objetivo:** pasar de un enunciado a variables, dominios y restricciones; resolverlo con CP-SAT e interpretar y verificar la respuesta.
El alcance es **todos los conceptos utilizados en los dos notebooks**, con fundamentos y ampliaciones breves; no es un catálogo de todos los módulos de OR-Tools.

## Ruta de repaso
1. Instalación, CSP, dominios y ciclo de resolución.
2. Booleanos, ALO/AMO/EO, implicación, equivalencia y conteos.
3. Productos, tablas de tuplas y `AllDifferent`.
4. Callbacks y enumeración limitada.
5. Caso de fútbol: modelo original, simplificación y extensión a 48 equipos.
6. Caso de texto: corpus, vocabulario, transiciones, rimas y restricciones.
7. Optimización, depuración, ejercicios resueltos y guía de métodos.

**Convención:** empleamos nombres `snake_case`. Al final se relacionan con los nombres `CamelCase` de tus notebooks. Los sorteos son ejercicios didácticos: no reproducen todas las reglas ni los bombos oficiales de FIFA.

## 1. Preparación
OR-Tools es una biblioteca de optimización. Aquí usaremos **CP-SAT**, su solucionador para modelos con variables enteras y booleanas. No es un modelo de lenguaje ni aprende por entrenamiento.

Se fija una versión para que el material sea reproducible. La instalación requiere internet una vez. Si Colab solicita reiniciar la sesión, reiníciala y vuelve a ejecutar desde la importación.

In [1]:
%pip -q install ortools==9.14.6206

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.26.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.31.1 which is incompatible.
googleapis-common-protos 1.75.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.31.1 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.31.1 which is incompatible.
google-cloud-dataproc 5.30.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.31.1 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.31.1 which is incompatible.
google-cloud-spanner 

In [2]:
import ortools
from ortools.sat.python import cp_model
from collections import Counter, defaultdict
from pathlib import Path
import random
import re
import unicodedata

print("Versión de OR-Tools:", ortools.__version__)
FACTIBLES = (cp_model.OPTIMAL, cp_model.FEASIBLE)

def resolver(modelo, segundos=10):
    """Valida la estructura y busca una solución; no imprime valores inexistentes."""
    error = modelo.validate()
    if error:
        raise ValueError(error)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = segundos
    solver.parameters.num_search_workers = 1
    solver.parameters.random_seed = 7
    estado = solver.solve(modelo)
    print("Estado:", solver.status_name(estado))
    return solver, estado

Versión de OR-Tools: 9.14.6206


## 2. ¿Qué es un CSP?
Un problema de satisfacción de restricciones (CSP) tiene:

| Componente | Pregunta | Ejemplo |
|---|---|---|
| Variable | ¿Qué debo decidir? | Grupo de un equipo |
| Dominio | ¿Qué valores puede tomar? | Grupos A a H |
| Restricción | ¿Qué debe cumplirse? | Cuatro equipos por grupo |
| Solución | ¿Qué asignación cumple todo? | Un sorteo válido |

**Satisfacción:** encontrar una solución válida. **Optimización:** elegir la mejor según una función objetivo.
La **aridad** es la cantidad de variables involucradas: `x == 2` es unaria; `x < y`, binaria; una tabla de cuatro posiciones, cuaternaria. `AllDifferent` es una restricción global.

CP-SAT combina propagación de restricciones y búsqueda con técnicas SAT. Declarar una restricción no asigna inmediatamente un valor a la variable. El solver encuentra una asignación compatible con todas las reglas.

### Primer modelo: dos números
Buscamos $x,y\in\{0,1,2,3,4,5\}$ tales que $x+y=5$ y $x<y$.
1. Crear el modelo.
2. Crear variables con límites inclusivos.
3. Añadir reglas.
4. Resolver.
5. Leer valores solo si se encontró solución.

In [3]:
m = cp_model.CpModel()                     # 1. Contenedor del modelo
x = m.new_int_var(0, 5, "x")               # 2. Decisión entera, no entero Python
y = m.new_int_var(0, 5, "y")
m.add(x + y == 5)                          # 3. Restricciones simultáneas
m.add(x < y)
s, estado = resolver(m)                    # 4. Búsqueda
if estado in FACTIBLES:                    # 5. Lectura segura
    print("x =", s.value(x), "y =", s.value(y))
    assert s.value(x) + s.value(y) == 5
    assert s.value(x) < s.value(y)

Estado: OPTIMAL
x = 0 y = 5


**Lectura:** las parejas válidas son `(0,5)`, `(1,4)` y `(2,3)`. El solver puede devolver cualquiera; no se pidió minimizar `x`.

| Estado | Interpretación |
|---|---|
| `OPTIMAL` | Con objetivo, se probó optimalidad; sin objetivo, puede indicar éxito de la búsqueda de factibilidad. |
| `FEASIBLE` | Hay solución, pero no se completó la prueba de optimalidad o la búsqueda solicitada. |
| `INFEASIBLE` | Se demostró que las restricciones son incompatibles. |
| `UNKNOWN` | No se pudo concluir; por ejemplo, se agotó el tiempo sin solución. |
| `MODEL_INVALID` | El modelo no cumple requisitos estructurales del solver. |

**No confundas:** `UNKNOWN` no demuestra imposibilidad. `OPTIMAL` en una búsqueda de una solución no significa que se hayan enumerado todas.

Las restricciones lineales de CP-SAT usan coeficientes enteros. Para `0.5*x + 1.5*y <= 4`, multiplica toda la desigualdad por 2: `x + 3*y <= 8`. Para precios, usa céntimos como unidad.

## 3. Booleanos y reglas de selección
`new_bool_var` crea una decisión con dominio `{0,1}`. Por ejemplo: 1 si un alumno participa y 0 si no participa.

| Regla | Ecuación | Método |
|---|---|---|
| ALO: al menos uno | `sum(b) >= 1` | `add_bool_or(b)` |
| AMO: a lo sumo uno | `sum(b) <= 1` | `add_at_most_one(b)` |
| EO: exactamente uno | `sum(b) == 1` | `add_exactly_one(b)` |

AMO permite que ninguno sea elegido; EO no lo permite. En tus grupos, cada equipo debe cumplir EO entre todos los grupos.

In [4]:
m = cp_model.CpModel()
b = [m.new_bool_var(f"opcion_{i}") for i in range(3)]
m.add_exactly_one(b)
m.add(b[1] == 0)  # La segunda opción no está disponible
s, estado = resolver(m)
if estado in FACTIBLES:
    print([s.value(v) for v in b])  # Un solo 1; la posición 1 vale 0

Estado: OPTIMAL
[1, 0, 0]


## 4. Implicación, reificación y channeling
`m.add(x < y).only_enforce_if(b)` expresa $b\Rightarrow(x<y)$.
Si `b=1`, la regla se exige. Si `b=0`, esta regla no impone nada sobre `x` e `y`.

| b | x < y | ¿Cumple la implicación? |
|---|---|---|
| 0 | Falso | Sí |
| 0 | Verdadero | Sí |
| 1 | Falso | No |
| 1 | Verdadero | Sí |

Para expresar **equivalencia**, $b\Leftrightarrow(x<y)$, hacen falta ambas direcciones. Es cómodo escribir:
- `b → x < y`.
- `¬b → x >= y`.

Este vínculo entre representaciones se llama **channeling**. La primera dirección sola es reificación parcial; ambas logran reificación completa. `~b` representa la negación de la variable booleana.

No escribas `if x < y:` para decisiones del modelo: un `if` de Python se evalúa al construir el modelo, cuando todavía no conoces sus valores.

In [5]:
for valor_x, valor_y in [(2, 5), (5, 2), (3, 3)]:
    m = cp_model.CpModel()
    x = m.new_int_var(0, 10, "x")
    y = m.new_int_var(0, 10, "y")
    b = m.new_bool_var("x_menor_y")
    m.add(x < y).only_enforce_if(b)
    m.add(x >= y).only_enforce_if(~b)
    m.add(x == valor_x)
    m.add(y == valor_y)
    s, estado = resolver(m)
    if estado in FACTIBLES:
        print(valor_x, valor_y, "→ b =", s.value(b))

Estado: OPTIMAL
2 5 → b = 1
Estado: OPTIMAL
5 2 → b = 0
Estado: OPTIMAL
3 3 → b = 0


**Resultado esperado:** `b` vale 1, 0 y 0. Para enteros, la negación de `x < y` es `x >= y`, no `x > y`.

`only_enforce_if([b1, b2])` activa una restricción si **ambos** literales son verdaderos. Para implicar un booleano a partir de otro, existe `add_implication(b1, b2)`. No todas las clases de restricciones admiten condiciones de activación: valida el modelo y consulta la API.

## 5. Contar valores: la función `count_vars`
Queremos contar cuántas variables valen `valor`.
1. Para cada variable `v`, crear un indicador booleano.
2. Vincularlo mediante **equivalencia**: vale 1 exactamente cuando `v == valor`.
3. Sumar los indicadores.

Una sola implicación permitiría que una coincidencia tuviera indicador 0 y se contara de menos. El dominio correcto del contador es `0..len(variables)`; no hace falta `0..1000`.

In [6]:
def count_vars(modelo, variables, valor, nombre):
    indicadores = []
    for i, v in enumerate(variables):
        coincide = modelo.new_bool_var(f"{nombre}_es_{i}")
        modelo.add(v == valor).only_enforce_if(coincide)
        modelo.add(v != valor).only_enforce_if(~coincide)
        indicadores.append(coincide)
    contador = modelo.new_int_var(0, len(variables), nombre)
    modelo.add(contador == sum(indicadores))
    return contador

m = cp_model.CpModel()
datos = [1, 2, 3, 4, 1, 5, 8, 1]
variables = [m.new_int_var(v, v, f"dato_{i}") for i, v in enumerate(datos)]
c = count_vars(m, variables, 1, "cantidad_de_unos")
s, estado = resolver(m)
if estado in FACTIBLES:
    print("Cantidad:", s.value(c))
    assert s.value(c) == 3

Estado: OPTIMAL
Cantidad: 3


## 6. Multiplicación y variables auxiliares
En el notebook de fútbol, `grupo_federacion[i][j] = federacion[i] * grupo[i][j]`.
La auxiliar vale 0 si el equipo no pertenece al grupo y el código de federación si pertenece. Por eso luego se pueden contar los códigos distintos de cero.

`add_multiplication_equality(z, [x, y])` declara `z = x*y`. Si un factor es una constante conocida, basta una igualdad lineal: `m.add(z == 3*b)`.
Los códigos 1, 2, 3… son **etiquetas**, no puntajes de calidad.

In [7]:
m = cp_model.CpModel()
fed = m.new_int_var(1, 5, "federacion")
pertenece = m.new_bool_var("pertenece")
aux = m.new_int_var(0, 5, "federacion_en_grupo")
m.add(fed == 3)
m.add(pertenece == 1)  # Cambia a 0 y vuelve a ejecutar esta celda
m.add_multiplication_equality(aux, [fed, pertenece])
s, estado = resolver(m)
if estado in FACTIBLES:
    print("Auxiliar:", s.value(aux))  # 3 si pertenece; 0 en caso contrario

Estado: OPTIMAL
Auxiliar: 3


## 7. Tablas permitidas, prohibidas y `AllDifferent`
Una tabla enumera combinaciones de valores. Es una restricción **extensional**: declara las tuplas explícitamente.

- `add_allowed_assignments([x,y], [(0,1),(1,2)])`: permite únicamente esas parejas, respetando el orden de las variables.
- `add_forbidden_assignments([x,y], [(1,2)])`: excluye esa pareja.
- `add_all_different([x,y,z])`: exige valores distintos entre todas las variables.

Cada fila de una tabla debe tener la misma longitud que la lista de variables. Una tabla permitida vacía hace imposible esa restricción; una prohibida vacía no excluye nada.

In [8]:
m = cp_model.CpModel()
x = m.new_int_var(0, 2, "x")
y = m.new_int_var(0, 2, "y")
z = m.new_int_var(0, 2, "z")
m.add_allowed_assignments([x, y], [(0, 1), (1, 2)])
m.add_forbidden_assignments([x, y], [(1, 2)])
m.add_all_different([x, y, z])
s, estado = resolver(m)
if estado in FACTIBLES:
    print([s.value(v) for v in [x, y, z]])  # [0, 1, 2]

Estado: OPTIMAL
[0, 1, 2]


## 8. Enumerar soluciones con un callback
Un **callback** es una función que el solver llama cuando encuentra una solución. La clase hereda de `CpSolverSolutionCallback`; `super().__init__()` inicializa la clase base; `self` permite guardar el contador y las variables entre llamadas.

Dentro del callback, se usa `self.value(v)` para leer la solución actual. `stop_search()` detiene la búsqueda. En un problema de satisfacción, `enumerate_all_solutions=True` pide enumerar; aquí usamos un solo trabajador y límites explícitos.

**Una muestra limitada no es el conjunto completo.** Si se alcanza el límite de tiempo o de soluciones, no afirmamos haber encontrado todas. La enumeración de grupos grandes puede ser enorme.

In [9]:
class MostrarSoluciones(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables, limite=5, formato=None):
        super().__init__()
        self.variables = variables
        self.limite = limite
        self.formato = formato
        self.cantidad = 0

    def on_solution_callback(self):
        self.cantidad += 1
        valores = [self.value(v) for v in self.variables]
        print(f"Solución {self.cantidad}:")
        print(self.formato(valores) if self.formato else valores)
        if self.cantidad >= self.limite:
            self.stop_search()

def enumerar(modelo, variables, limite=5, formato=None):
    error = modelo.validate()
    if error:
        raise ValueError(error)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.num_search_workers = 1
    solver.parameters.max_time_in_seconds = 5
    callback = MostrarSoluciones(variables, limite, formato)
    estado = solver.solve(modelo, callback)
    print("Estado:", solver.status_name(estado), "| Observadas:", callback.cantidad)
    return callback.cantidad

m = cp_model.CpModel()
x = m.new_int_var(0, 5, "x")
y = m.new_int_var(0, 5, "y")
m.add(x + y == 5)
m.add(x < y)
cantidad = enumerar(m, [x, y], limite=10)
assert cantidad == 3

Solución 1:
[0, 5]
Solución 2:
[1, 4]
Solución 3:
[2, 3]
Estado: OPTIMAL | Observadas: 3


## 9. Caso 1: formación de grupos de fútbol
### 9.1 Traducir el enunciado
Usaremos $g_{ij}\in\{0,1\}$: vale 1 si el equipo $i$ está en el grupo $j$.

| Regla | Ecuación |
|---|---|
| Un grupo por equipo | $\sum_j g_{ij}=1$ |
| Cuatro equipos por grupo | $\sum_i g_{ij}=4$ |
| Un equipo de cada bombo | $\sum_{i:b_i=k}g_{ij}=1$ |
| Como máximo dos UEFA | $\sum_{i:f_i=\mathrm{UEFA}}g_{ij}\leq2$ |
| Como máximo uno de otra federación | $\sum_{i:f_i=f}g_{ij}\leq1$ |

La **fila** de la matriz recorre grupos para un equipo; la **columna** recorre equipos para un grupo. Para 32 equipos y 8 grupos hay 256 decisiones booleanas.

### 9.2 Datos del notebook 6
Conservamos los equipos y la asignación didáctica de bombos de 2022 del archivo. Federación y bombo son datos conocidos, por lo que se guardan como enteros o cadenas de Python, sin pedir al solver que los decida.

In [10]:
equipos22 = ['Alemania', 'Francia', 'Países Bajos', 'Gales', 'Polonia', 'Portugal', 'Croacia', 'Inglaterra', 'Bélgica', 'Dinamarca', 'Serbia', 'Suiza', 'España', 'Catar', 'Arabia Saudita', 'Corea del Sur', 'Australia', 'Irán', 'Japón', 'Ghana', 'Senegal', 'Camerún', 'Marruecos', 'Túnez', 'Argentina', 'Brasil', 'Ecuador', 'Uruguay', 'Costa Rica', 'Canadá', 'Estados Unidos', 'México']
federaciones22 = ["UEFA"]*13 + ["AFC"]*6 + ["CAF"]*5 + ["Conmebol"]*4 + ["Concacaf"]*4
indices_bombos22 = [
    [0, 1, 2, 3, 4, 13, 23, 24],
    [5, 6, 7, 8, 9, 25, 27, 28],
    [10, 11, 14, 15, 16, 18, 19, 20]
]
bombos22 = [4] * len(equipos22)
for numero, indices in enumerate(indices_bombos22, start=1):
    for i in indices:
        bombos22[i] = numero
print("Equipos:", len(equipos22))
print("Federaciones:", Counter(federaciones22))
print("Bombos:", Counter(bombos22))

Equipos: 32
Federaciones: Counter({'UEFA': 13, 'AFC': 6, 'CAF': 5, 'Conmebol': 4, 'Concacaf': 4})
Bombos: Counter({1: 8, 2: 8, 3: 8, 4: 8})


### 9.3 Construcción del modelo, paso a paso
La función crea un modelo nuevo en cada llamada: así no se acumulan restricciones de ejecuciones anteriores.

1. Validar los datos antes de llamar al solver.
2. Crear `g[i][j]` con listas anidadas.
3. Aplicar exactamente uno a cada fila.
4. Sumar cada columna para obtener el tamaño del grupo.
5. Filtrar índices por federación o bombo y sumar solo las decisiones correspondientes.

En `sum(g[i][j] for i in range(n) if federaciones[i] == fed)`, el `if` sí es correcto: compara **datos de Python**, no variables por resolver.

In [11]:
def crear_grupos(equipos, federaciones, bombos, numero_grupos):
    n = len(equipos)
    assert n == 4 * numero_grupos, "Se necesitan cuatro equipos por grupo"
    assert len(federaciones) == len(bombos) == n
    assert len(set(equipos)) == n, "No repitas equipos"
    assert set(bombos) == {1, 2, 3, 4}
    assert all(bombos.count(k) == numero_grupos for k in range(1, 5))
    for fed, cantidad in Counter(federaciones).items():
        capacidad = (2 if fed == "UEFA" else 1) * numero_grupos
        assert cantidad <= capacidad, f"Demasiados equipos de {fed}"

    m = cp_model.CpModel()
    g = [[m.new_bool_var(f"g_{i}_{j}") for j in range(numero_grupos)]
         for i in range(n)]

    for i in range(n):
        m.add_exactly_one(g[i])

    for j in range(numero_grupos):
        m.add(sum(g[i][j] for i in range(n)) == 4)
        for fed in sorted(set(federaciones)):
            limite = 2 if fed == "UEFA" else 1
            m.add(sum(g[i][j] for i in range(n)
                      if federaciones[i] == fed) <= limite)
        for k in range(1, 5):
            m.add(sum(g[i][j] for i in range(n)
                      if bombos[i] == k) == 1)
    return m, g

m22, g22 = crear_grupos(equipos22, federaciones22, bombos22, 8)
s22, estado22 = resolver(m22)

Estado: OPTIMAL


### 9.4 Leer y verificar la solución
No basta con imprimir: reconstruimos cada grupo y comprobamos las reglas con Python. Estos chequeos verifican el resultado sin depender de las expresiones que contiene el modelo.
`Counter` cuenta etiquetas conocidas después de resolver; `count_vars` cuenta coincidencias de decisiones dentro del modelo.

In [12]:
def mostrar_y_verificar(solver, estado, g, equipos, federaciones, bombos):
    if estado not in FACTIBLES:
        print("No hay una solución disponible para mostrar.")
        return
    asignados = []
    for j in range(len(g[0])):
        indices = [i for i in range(len(equipos)) if solver.value(g[i][j])]
        assert len(indices) == 4
        assert sorted(bombos[i] for i in indices) == [1, 2, 3, 4]
        conteo = Counter(federaciones[i] for i in indices)
        assert all(c <= (2 if f == "UEFA" else 1) for f, c in conteo.items())
        print(f"Grupo {chr(65+j)}:")
        for i in sorted(indices, key=lambda i: bombos[i]):
            print(f"  {equipos[i]:22} | {federaciones[i]:9} | bombo {bombos[i]}")
        asignados.extend(indices)
    assert sorted(asignados) == list(range(len(equipos)))
    print("Verificación completa: todos los equipos aparecen exactamente una vez.")

mostrar_y_verificar(s22, estado22, g22, equipos22, federaciones22, bombos22)

Grupo A:
  Túnez                  | CAF       | bombo 1
  Uruguay                | Conmebol  | bombo 2
  Japón                  | AFC       | bombo 3
  México                 | Concacaf  | bombo 4
Grupo B:
  Países Bajos           | UEFA      | bombo 1
  Croacia                | UEFA      | bombo 2
  Australia              | AFC       | bombo 3
  Estados Unidos         | Concacaf  | bombo 4
Grupo C:
  Argentina              | Conmebol  | bombo 1
  Inglaterra             | UEFA      | bombo 2
  Corea del Sur          | AFC       | bombo 3
  Canadá                 | Concacaf  | bombo 4
Grupo D:
  Alemania               | UEFA      | bombo 1
  Costa Rica             | Concacaf  | bombo 2
  Serbia                 | UEFA      | bombo 3
  Ecuador                | Conmebol  | bombo 4
Grupo E:
  Francia                | UEFA      | bombo 1
  Bélgica                | UEFA      | bombo 2
  Arabia Saudita         | AFC       | bombo 3
  Marruecos              | CAF       | bombo 4
Grupo F:
  Gale

### 9.5 Recuperar la formulación original con auxiliares
La versión original fija variables `federacion[i]` y `bombo[i]`, las multiplica por `g[i][j]` y cuenta sus resultados. Es válida, aunque introduce más variables.

Esta celda reconstruye ese procedimiento para el grupo A dentro del modelo anterior. Los conteos deben coincidir con los de la formulación simplificada. **Añadir estas auxiliares no cambia las reglas del sorteo.**

In [13]:
codigos = {f: k+1 for k, f in enumerate(sorted(set(federaciones22)))}
aux_fed, aux_bombo = [], []
for i in range(len(equipos22)):
    f = m22.new_int_var(1, 5, f"f_{i}")
    b = m22.new_int_var(1, 4, f"b_{i}")
    m22.add(f == codigos[federaciones22[i]])
    m22.add(b == bombos22[i])
    af = m22.new_int_var(0, 5, f"af_{i}")
    ab = m22.new_int_var(0, 4, f"ab_{i}")
    m22.add_multiplication_equality(af, [f, g22[i][0]])
    m22.add_multiplication_equality(ab, [b, g22[i][0]])
    aux_fed.append(af)
    aux_bombo.append(ab)
conteos_f = {f: count_vars(m22, aux_fed, k, f"cuenta_{f}") for f, k in codigos.items()}
conteos_b = {k: count_vars(m22, aux_bombo, k, f"cuenta_b{k}") for k in range(1, 5)}
s22, estado22 = resolver(m22)
if estado22 in FACTIBLES:
    print("Federaciones en A:", {f: s22.value(c) for f, c in conteos_f.items()})
    print("Bombos en A:", {k: s22.value(c) for k, c in conteos_b.items()})
    indices = [i for i in range(32) if s22.value(g22[i][0])]
    assert all(s22.value(c) == sum(federaciones22[i] == f for i in indices)
               for f, c in conteos_f.items())
    assert all(s22.value(c) == 1 for c in conteos_b.values())

Estado: OPTIMAL
Federaciones en A: {'AFC': 1, 'CAF': 1, 'Concacaf': 1, 'Conmebol': 1, 'UEFA': 0}
Bombos en A: {1: 1, 2: 1, 3: 1, 4: 1}


### 9.6 Extender a 48 equipos: revisar datos y dimensiones
El bloque «Mundial 2026» del notebook contiene 48 nombres. Aquí lo usamos como **escenario de práctica**, sin afirmar que listas y bombos sean los oficiales.

Correcciones respecto del código adjunto:

| Hallazgo | Consecuencia | Corrección |
|---|---|---|
| Conteos con `range(32)` | Se ignoran 16 equipos | Usar `len(equipos)` |
| `else` del bombo 4 comentado | Algunos bombos quedan como decisiones libres | Fijar todos los bombos |
| Federación asignada por tramos con Argelia entre AFC | Etiqueta incorrecta | Mover Argelia a CAF en los datos |
| Comentario «cuatro federaciones distintas» | No describe la excepción UEFA | Como máximo 2 UEFA y 1 de cada otra |
| Dominio `0..1000` | Rango innecesario | Ajustar límites a cada magnitud |
| Solo se acepta `OPTIMAL` | Se descartan soluciones `FEASIBLE` | Comprobar ambos estados |

Los 12 integrantes de cada bombo se conservan según los índices del ejercicio y su complemento. Una validación de cantidades es necesaria, pero no garantiza por sí sola que todas las reglas sean compatibles.

In [14]:
equipos48 = ['Alemania', 'Inglaterra', 'Portugal', 'República Checa', 'Croacia', 'Austria', 'Bélgica', 'Suecia', 'Bosnia y Herzegovina', 'Escocia', 'Suiza', 'España', 'Noruega', 'Turquía', 'Francia', 'Países Bajos', 'Arabia Saudita', 'Corea del Sur', 'Argelia', 'Irak', 'Irán', 'Australia', 'Japón', 'Jordania', 'Catar', 'Uzbekistán', 'Costa de Marfil', 'R. D. del Congo', 'Senegal', 'Sudáfrica', 'Egipto', 'Marruecos', 'Túnez', 'Cabo Verde', 'Ghana', 'Colombia', 'Paraguay', 'Argentina', 'Ecuador', 'Brasil', 'Uruguay', 'Haití', 'Curazao', 'México', 'Estados Unidos', 'Canadá', 'Panamá', 'Nueva Zelanda']
federaciones48 = ["UEFA"]*16 + ["AFC"]*10 + ["CAF"]*9 + ["Conmebol"]*6 + ["Concacaf"]*6 + ["OFC"]
federaciones48[equipos48.index("Argelia")] = "CAF"
indices_bombos48 = [
    [43, 44, 45, 11, 37, 1, 14, 39, 2, 15, 0, 6],
    [4, 35, 40, 31, 10, 22, 28, 20, 17, 38, 5, 21],
    [46, 30, 18, 12, 9, 36, 26, 32, 16, 24, 25, 29]
]
indices_fijados = [i for bloque in indices_bombos48 for i in bloque]
assert len(set(indices_fijados)) == len(indices_fijados)
bombos48 = [4] * len(equipos48)
for k, indices in enumerate(indices_bombos48, start=1):
    for i in indices:
        bombos48[i] = k
print("Federaciones del escenario:", Counter(federaciones48))
print("Bombos:", Counter(bombos48))
m48, g48 = crear_grupos(equipos48, federaciones48, bombos48, 12)
s48, estado48 = resolver(m48)
mostrar_y_verificar(s48, estado48, g48, equipos48, federaciones48, bombos48)

Federaciones del escenario: Counter({'UEFA': 16, 'CAF': 10, 'AFC': 9, 'Conmebol': 6, 'Concacaf': 6, 'OFC': 1})
Bombos: Counter({1: 12, 4: 12, 2: 12, 3: 12})
Estado: OPTIMAL
Grupo A:
  Alemania               | UEFA      | bombo 1
  Senegal                | CAF       | bombo 2
  Catar                  | AFC       | bombo 3
  Nueva Zelanda          | OFC       | bombo 4
Grupo B:
  Canadá                 | Concacaf  | bombo 1
  Ecuador                | Conmebol  | bombo 2
  Noruega                | UEFA      | bombo 3
  Ghana                  | CAF       | bombo 4
Grupo C:
  Estados Unidos         | Concacaf  | bombo 1
  Japón                  | AFC       | bombo 2
  Paraguay               | Conmebol  | bombo 3
  Cabo Verde             | CAF       | bombo 4
Grupo D:
  México                 | Concacaf  | bombo 1
  Colombia               | Conmebol  | bombo 2
  Arabia Saudita         | AFC       | bombo 3
  Suecia                 | UEFA      | bombo 4
Grupo E:
  Bélgica                | UEF

### 9.7 Simetría y muestra de sorteos
Si no hay reglas que distingan A de B, intercambiar los nombres de esos grupos produce otra asignación equivalente. Esto se llama **simetría**.
Fijar el primer equipo en A elimina parte de esa simetría. Es válido aquí porque los grupos tienen las mismas reglas; no siempre lo será si algunos grupos tienen sedes o restricciones especiales.

Enumeramos solamente dos soluciones. Aplanamos la matriz por filas y recuperamos cada posición con `i*numero_grupos+j`.

In [15]:
m_muestra, g_muestra = crear_grupos(equipos22, federaciones22, bombos22, 8)
m_muestra.add(g_muestra[0][0] == 1)
def formato_grupos(valores):
    return "\n".join(
        f"{chr(65+j)}: " + ", ".join(equipos22[i] for i in range(32) if valores[i*8+j])
        for j in range(8))
enumerar(m_muestra, [v for fila in g_muestra for v in fila],
         limite=2, formato=formato_grupos)

Solución 1:
A: Alemania, Suiza, Uruguay, México
B: Corea del Sur, Túnez, Brasil, Estados Unidos
C: Croacia, Serbia, Argentina, Canadá
D: Francia, Australia, Ecuador, Costa Rica
E: Países Bajos, Bélgica, Japón, Marruecos
F: Polonia, Inglaterra, Arabia Saudita, Camerún
G: Portugal, España, Catar, Senegal
H: Gales, Dinamarca, Irán, Ghana
Solución 2:
A: Alemania, España, Senegal, Uruguay
B: Inglaterra, Corea del Sur, Túnez, México
C: Suiza, Camerún, Argentina, Costa Rica
D: Francia, Australia, Brasil, Estados Unidos
E: Países Bajos, Portugal, Japón, Ecuador
F: Polonia, Dinamarca, Arabia Saudita, Canadá
G: Bélgica, Serbia, Catar, Marruecos
H: Gales, Croacia, Irán, Ghana
Estado: FEASIBLE | Observadas: 2


2

## 10. Caso 2: generar texto con restricciones
### 10.1 Del texto a un problema discreto
El notebook 7 utiliza palabras como valores de decisiones. CP-SAT trabaja con enteros: asignamos un ID a cada palabra y recuperamos el texto al terminar.

| Objeto | Significado |
|---|---|
| `words_order` | Palabras del corpus en orden, con repeticiones |
| `words` | Vocabulario sin repeticiones |
| `word_to_i` | Diccionario palabra → ID |
| `seq[p]` | ID de la palabra elegida para la posición p |
| `transition_table` | Parejas consecutivas observadas |
| `rhymes_table` | Parejas con terminaciones compatibles |
| `plagiarism_table` | Ventanas observadas de cuatro palabras que se prohíben |

El archivo `vallejo.txt` no fue adjuntado. Incluimos un corpus didáctico propio. Puedes reemplazarlo por un `.txt` UTF-8: cambiar el corpus cambia las soluciones y puede volver el modelo imposible.

In [16]:
# Corpus didáctico propio formado por fragmentos de dos palabras.
# Repetir la palabra de enlace entre fragmentos permite estudiar bigramas
# sin que una ruta recombinada copie ventanas completas de cuatro palabras.
lineas_base = [
    "clara luna mira suave canción",
    "fría brisa toca tibia pasión",
    "joven nube sueña leve oración",
    "alta estrella busca fresca emoción",
]
palabras_base = " ".join(lineas_base).split()
fragmentos = [f"{a} {b}" for a, b in zip(palabras_base, palabras_base[1:])]
fragmentos += ["luna toca", "brisa mira", "nube busca", "estrella sueña"]
corpus_demo = "\n".join(fragmentos)
ruta_corpus = None  # Ejemplo: "/content/vallejo.txt" después de subirlo a Colab
text = Path(ruta_corpus).read_text(encoding="utf-8") if ruta_corpus else corpus_demo
text = unicodedata.normalize("NFC", text)
words_order = re.findall(r"\w+", text.lower())
words = sorted(set(words_order))
assert words_order, "El corpus no puede estar vacío"
word_to_i = {w: i for i, w in enumerate(words)}
print("Palabras totales:", len(words_order), "| Vocabulario:", len(words))
print(list(word_to_i.items())[:8])

Palabras totales: 46 | Vocabulario: 20
[('alta', 0), ('brisa', 1), ('busca', 2), ('canción', 3), ('clara', 4), ('emoción', 5), ('estrella', 6), ('fresca', 7)]


**Paso a paso:** `lower()` unifica mayúsculas; `findall` extrae secuencias de caracteres de palabra; `set` elimina duplicados; `sorted` da IDs estables; `enumerate` produce parejas `(índice, palabra)`.
`\w+` también admite números y guiones bajos: es tokenización sencilla, no análisis lingüístico completo.

### 10.2 Azar sin restricciones
`random.choice(words)` selecciona una palabra de forma segura. En el original, `randint(0,len(words))` podía devolver `len(words)`, que está fuera de los índices válidos.
`random.sample(lista, k)` muestrea sin repetición dentro de esa llamada, pero llamadas separadas pueden repetir valores.

In [17]:
rng = random.Random(7)
print(" ".join(rng.choice(words) for _ in range(10)))

leve clara mira brisa busca sueña canción luna tibia brisa


### 10.3 Agrupar terminaciones: aproximación a la rima
Reproducimos la heurística del notebook con las vocales acentuadas añadidas:
1. Ignorar palabras de menos de cuatro caracteres.
2. Inspeccionar sus últimas letras.
3. Usar normalmente los tres últimos caracteres como clave; conservar el caso especial `*` del original.
4. Conservar grupos de tres palabras o más.

Es una **heurística ortográfica**, no un detector fonético de rima consonante o asonante. No analiza acentuación ni pronunciación. El grupo mínimo de tres es una decisión del ejercicio, no una condición lingüística.

In [18]:
def clave_rima(palabra):
    if len(palabra) < 4:
        return None
    vocales = "aeiouáéíóúü"
    final = palabra[-2:]
    if palabra[-3] in vocales:
        final = palabra[-3] + final
    elif palabra[-2] in vocales:
        final = palabra[-3] + final
    elif palabra[-2] in "rl":
        final = "*" + final
    else:
        final = palabra[-3] + final
    return final

rhyme_groups = defaultdict(list)
for w in words:
    clave = clave_rima(w)
    if clave is not None:
        rhyme_groups[clave].append(w)
rhyme_groups = {k: v for k, v in rhyme_groups.items() if len(v) >= 3}
print(rhyme_groups)

if rhyme_groups:
    for _ in range(2):  # Dos pares de versos: cuatro versos en total
        clave = rng.choice(sorted(rhyme_groups))
        for _ in range(2):
            verso = [rng.choice(words) for _ in range(4)]
            verso.append(rng.choice(rhyme_groups[clave]))
            print(" ".join(verso))
else:
    print("No hay grupos de rima suficientes.")

{'ión': ['canción', 'emoción', 'oración', 'pasión']}
brisa busca nube nube canción
fresca busca sueña nube canción
fresca tibia brisa tibia pasión
brisa fresca brisa sueña emoción


**Observa:** los finales son compatibles según la heurística, pero el interior de los versos es aleatorio. Tampoco se obliga aún a usar finales distintos.

### 10.4 Transiciones y recorrido aleatorio
Una transición `a → b` existe si `b` apareció después de `a` en el corpus. Creamos listas de sucesores y eliminamos duplicados.
El resultado es un grafo dirigido de palabras. Elegir uniformemente entre sucesores distintos **no conserva las frecuencias** del corpus.

Como en el original, se tokeniza el texto entero: se incluyen transiciones entre líneas. Si deseas respetar límites de oración, construye las parejas por oración.

In [19]:
transition = defaultdict(set)
for actual, siguiente in zip(words_order, words_order[1:]):
    transition[actual].add(siguiente)
transition = {w: sorted(siguientes) for w, siguientes in transition.items()}
for w in list(transition)[:6]:
    print(w, "→", transition[w])

secuencia_azar = [rng.choice(sorted(transition))] if transition else []
for _ in range(9):
    siguientes = transition.get(secuencia_azar[-1], [])
    if not siguientes:
        print("Recorrido detenido: la última palabra no tiene sucesores.")
        break
    secuencia_azar.append(rng.choice(siguientes))
print(" ".join(secuencia_azar))

clara → ['luna']
luna → ['luna', 'mira', 'toca']
mira → ['mira', 'nube', 'suave']
suave → ['canción', 'suave']
canción → ['canción', 'fría']
fría → ['brisa', 'fría']
joven nube busca fresca emoción luna toca toca brisa brisa


**Limitación del azar:** escoger una palabra válida ahora puede conducir a un callejón sin salida después. El solver busca una secuencia que satisfaga conjuntamente transiciones, finales y restricciones de repetición.

### 10.5 Construir las tablas
La tabla de transiciones contiene IDs de parejas. La de rimas usa el producto cartesiano de cada grupo: todas las parejas de ese grupo, incluso `(w,w)` como en el original. Más adelante `AllDifferent` excluye usar la misma palabra dos veces.

La tabla llamada `plagiarism_table` en el original contiene cada ventana consecutiva de cuatro palabras del corpus. Prohibirla evita copiar esas ventanas exactas: **no es un detector ni una garantía de ausencia de plagio**.

In [20]:
transition_table = sorted({(word_to_i[a], word_to_i[b])
                           for a, siguientes in transition.items() for b in siguientes})
rhymes_table = sorted({(word_to_i[a], word_to_i[b])
                      for grupo in rhyme_groups.values() for a in grupo for b in grupo})
plagiarism_table = sorted({tuple(word_to_i[w] for w in words_order[i:i+4])
                          for i in range(len(words_order)-3)})
print("Transiciones:", len(transition_table), transition_table[:5])
print("Parejas de rima:", len(rhymes_table))
print("Ventanas prohibidas:", len(plagiarism_table))

Transiciones: 45 [(0, 0), (0, 6), (1, 1), (1, 12), (1, 19)]
Parejas de rima: 16
Ventanas prohibidas: 43


### 10.6 Modelo completo, posición por posición
Generamos 20 palabras, distribuidas en cuatro líneas de cinco:
- Cada `seq[p]` tiene dominio `0..len(words)-1`.
- Hay 19 restricciones de transición, incluida la transición entre líneas.
- Los finales de las líneas están en índices 4, 9, 14 y 19, porque Python empieza en 0.
- Exigimos rima entre `(4,9)` y entre `(14,19)`: esquema AABB; no obligamos a que A y B sean rimas diferentes.
- Hay 17 ventanas de cuatro posiciones. Prohibimos las observadas en el corpus, incluidas las que cruzan líneas.
- `AllDifferent(seq)` impide repetir cualquier palabra en las 20 posiciones.

**Factibilidad:** tener al menos 20 palabras distintas es necesario para `AllDifferent`, pero no suficiente para cumplir todo lo demás.

In [21]:
def crear_texto(usar_rimas=True, evitar_ventanas=True, todas_distintas=True):
    m = cp_model.CpModel()
    seq = [m.new_int_var(0, len(words)-1, f"seq_{p}") for p in range(20)]
    for p in range(19):
        m.add_allowed_assignments([seq[p], seq[p+1]], transition_table)
    if usar_rimas:
        m.add_allowed_assignments([seq[4], seq[9]], rhymes_table)
        m.add_allowed_assignments([seq[14], seq[19]], rhymes_table)
    if evitar_ventanas:
        for p in range(17):
            m.add_forbidden_assignments(seq[p:p+4], plagiarism_table)
    if todas_distintas:
        m.add_all_different(seq)
    return m, seq

def formato_poema(ids):
    return "\n".join(" ".join(words[i] for i in ids[p:p+5]) for p in range(0, len(ids), 5))

m_texto, seq = crear_texto()
s_texto, estado_texto = resolver(m_texto)
poema_generado = None
if estado_texto in FACTIBLES:
    ids = [s_texto.value(v) for v in seq]
    poema_generado = formato_poema(ids)
    print(poema_generado)
    assert len(set(ids)) == 20
    assert all(tuple(ids[p:p+2]) in set(transition_table) for p in range(19))
    assert (ids[4], ids[9]) in set(rhymes_table)
    assert (ids[14], ids[19]) in set(rhymes_table)
    assert all(tuple(ids[p:p+4]) not in set(plagiarism_table) for p in range(17))
    print("Todas las restricciones fueron verificadas.")
else:
    print("Revisa las restricciones por etapas en la sección siguiente.")

Estado: OPTIMAL
clara luna mira suave canción
fría brisa toca tibia pasión
joven nube sueña leve oración
alta estrella busca fresca emoción
Todas las restricciones fueron verificadas.


**Interpretación:** una secuencia válida cumple las reglas formales; no necesariamente posee significado, métrica o calidad literaria. El corpus incluido consiste en fragmentos de dos palabras construidos para mostrar recombinaciones factibles; no pretende representar una obra literaria natural. No se ha entrenado una IA generativa ni se evalúa originalidad semántica.

### 10.7 Guardar y enumerar
El archivo de texto se escribe solo si hay una solución. El contexto `with` cierra el archivo automáticamente; no hace falta `f.close()` dentro de él.
La enumeración se limita a tres ejemplos, con la misma clase de callback usada anteriormente.

In [22]:
if poema_generado is not None:
    with open("poema_generado.txt", "w", encoding="utf-8") as archivo:
        archivo.write(poema_generado + "\n")
    print("Archivo creado: poema_generado.txt")
    # En Colab, puedes descargarlo desde el panel Archivos.

enumerar(m_texto, seq, limite=3, formato=formato_poema)

Archivo creado: poema_generado.txt
Solución 1:
clara luna mira suave canción
fría brisa toca tibia pasión
joven nube sueña leve oración
alta estrella busca fresca emoción
Estado: OPTIMAL | Observadas: 1


1

### 10.8 Diagnóstico incremental
Construimos cuatro modelos nuevos: cada uno añade una familia de restricciones. Si una etapa es factible y la siguiente es `INFEASIBLE`, la incompatibilidad surge al combinar la nueva familia con las anteriores.
Esto orienta el diagnóstico, pero no identifica necesariamente un conjunto mínimo de restricciones en conflicto.
Si aparece `UNKNOWN`, aumenta el límite antes de concluir que hay incompatibilidad.

In [23]:
etapas = [
    ("Solo transiciones", False, False, False),
    ("+ rimas", True, False, False),
    ("+ prohibir ventanas", True, True, False),
    ("+ palabras distintas", True, True, True),
]
for nombre, rimas, ventanas, distintas in etapas:
    print(nombre)
    modelo_etapa, _ = crear_texto(rimas, ventanas, distintas)
    resolver(modelo_etapa, segundos=5)

Solo transiciones
Estado: OPTIMAL
+ rimas
Estado: OPTIMAL
+ prohibir ventanas
Estado: OPTIMAL
+ palabras distintas
Estado: OPTIMAL


### 10.9 Alternativa de repetición del notebook original
`AllDifferent(seq)` impone 20 palabras distintas en total. El bloque comentado del original propone una condición más débil: para cada columna de los cuatro versos, exigir cuatro palabras distintas.

Por ejemplo, los índices `[0,5,10,15]` son la primera palabra de cada verso. Esta variante permite repetir una palabra si aparece en columnas distintas.

In [24]:
m_columnas, seq_columnas = crear_texto(todas_distintas=False)
for columna in range(5):
    m_columnas.add_all_different([seq_columnas[fila*5+columna] for fila in range(4)])
s_columnas, estado_columnas = resolver(m_columnas)
if estado_columnas in FACTIBLES:
    print(formato_poema([s_columnas.value(v) for v in seq_columnas]))

Estado: OPTIMAL
busca estrella busca fresca emoción
luna luna mira suave canción
fría brisa toca tibia pasión
joven nube sueña leve oración


## 11. Ampliación: de satisfacción a optimización
Los dos notebooks buscan soluciones factibles. Podemos añadir un objetivo para comparar soluciones.
Ejemplo: elegir exactamente dos actividades con presupuesto máximo de 8. Cada actividad tiene un costo y una utilidad didáctica.

1. Booleano por actividad: seleccionar o no.
2. Restricciones de cantidad y costo.
3. `maximize` para sumar la utilidad elegida.
4. Interpretar `objective_value` solo si existe solución.

Las restricciones son obligatorias; el objetivo elige entre asignaciones que ya las cumplen. Maximizar no permite violar el presupuesto.

In [25]:
actividades = ["Taller de código", "Videojuegos", "Música", "Cine"]
costos = [3, 5, 4, 2]
utilidades = [7, 9, 6, 4]
m = cp_model.CpModel()
elegir = [m.new_bool_var(f"actividad_{i}") for i in range(4)]
m.add(sum(elegir) == 2)
m.add(sum(costos[i] * elegir[i] for i in range(4)) <= 8)
m.maximize(sum(utilidades[i] * elegir[i] for i in range(4)))
s, estado = resolver(m)
if estado in FACTIBLES:
    print("Elegidas:", [actividades[i] for i in range(4) if s.value(elegir[i])])
    print("Utilidad:", s.objective_value)
    print("Cota del objetivo:", s.best_objective_bound)
    assert s.objective_value == 16

Estado: OPTIMAL
Elegidas: ['Taller de código', 'Videojuegos']
Utilidad: 16.0
Cota del objetivo: 16.0


**Resultado esperado:** Taller de código y Videojuegos, costo 8 y utilidad 16. En maximización, la mejor cota es un límite superior para el valor óptimo; con optimalidad probada coincide con el objetivo.

No uses la enumeración exhaustiva anterior directamente sobre un modelo con objetivo. Para enumerar soluciones óptimas, primero demuestra el óptimo; después reconstruye el modelo sin objetivo, fija la puntuación a ese óptimo y enumera.

## 12. Errores frecuentes y cómo diagnosticarlos

| Síntoma | Posible causa | Acción |
|---|---|---|
| `INFEASIBLE` | Reglas incompatibles | Añadir familias de reglas por etapas |
| `UNKNOWN` | Límite insuficiente | Revisar tamaño y tiempo; no afirmar imposibilidad |
| Índice fuera de rango | Límite superior inclusivo de `randint` | Usar `choice` |
| `KeyError` en transiciones | Palabra sin sucesores | Usar `get` y detener el recorrido |
| Datos repetidos en sorteo | Falta exactamente uno | Revisar filas y columnas |
| Conteo demasiado pequeño | Solo una dirección de channeling | Añadir la negación complementaria |
| Poema ilegible pese a ser válido | Reglas locales insuficientes | Distinguir factibilidad de calidad lingüística |
| Resultados cambian | Varias soluciones o parámetros distintos | Validar reglas, no exigir una salida particular |
| Ejecución interminable | Enumeración enorme | Limitar tiempo y cantidad |

`model.validate()` revisa estructura, no demuestra factibilidad. `model.model_stats()` describe el modelo; `solver.response_stats()` describe la búsqueda. Evita imprimir matrices gigantes cuando una tabla de grupos comunica mejor el resultado.

In [26]:
# Contradicción intencional: no causa excepción, retorna INFEASIBLE.
m_imposible = cp_model.CpModel()
a = m_imposible.new_int_var(0, 3, "a")
m_imposible.add(a >= 2)
m_imposible.add(a <= 1)
s_imposible, e_imposible = resolver(m_imposible)
assert e_imposible == cp_model.INFEASIBLE
print(m_imposible.model_stats())
# Para inspeccionar la búsqueda en detalle:
# print(s_imposible.response_stats())

Estado: INFEASIBLE
satisfaction model '': (model_fingerprint: 0x103642b8375d5bda)
#Variables: 1 (1 primary variables)
  - 1 in [0,3]
#kLinear1: 2


## 13. Ejercicios de repaso
Intenta resolverlos antes de ejecutar las soluciones.

1. **Valores y dominios.** Busca `x,y` entre 0 y 6, con suma 6 y `x < y`. Enumera todas las parejas. ¿Cuántas son?
2. **Channeling.** Una nota está entre 0 y 20. Modela `aprobado ↔ nota >= 11`. Fija nota 10 y luego 11. ¿Qué cambia si solo incluyes una implicación?
3. **Conteo.** Crea cinco variables entre 1 y 3 y exige que exactamente dos valgan 3.
4. **Grupos.** En el caso de 32 equipos, fija Catar en A y exige que Argentina y Brasil estén en grupos diferentes.
5. **Texto.** Mantén las restricciones completas y fuerza `clara` como primera palabra. Trabaja con el corpus incluido. ¿Sigue habiendo solución?
6. **Reflexión.** Si un corpus tiene 15 palabras distintas, ¿puede generar 20 palabras con `AllDifferent`? ¿Y si la unicidad se exige solo por columnas?

### Solución 1 — Parejas

In [27]:
m = cp_model.CpModel()
x = m.new_int_var(0, 6, "x")
y = m.new_int_var(0, 6, "y")
m.add(x + y == 6)
m.add(x < y)
assert enumerar(m, [x, y], limite=10) == 3
# Parejas: (0,6), (1,5), (2,4).

Solución 1:
[0, 6]
Solución 2:
[1, 5]
Solución 3:
[2, 4]
Estado: OPTIMAL | Observadas: 3


### Solución 2 — Aprobación
La segunda restricción impide que una nota aprobatoria tenga indicador 0. La equivalencia produce 0 para nota 10 y 1 para nota 11.

In [28]:
for valor in [10, 11]:
    m = cp_model.CpModel()
    nota = m.new_int_var(0, 20, "nota")
    aprobado = m.new_bool_var("aprobado")
    m.add(nota >= 11).only_enforce_if(aprobado)
    m.add(nota <= 10).only_enforce_if(~aprobado)
    m.add(nota == valor)
    s, estado = resolver(m)
    if estado in FACTIBLES:
        print("Nota:", valor, "Aprobado:", s.value(aprobado))

Estado: OPTIMAL
Nota: 10 Aprobado: 0
Estado: OPTIMAL
Nota: 11 Aprobado: 1


### Solución 3 — Contar decisiones
Las cinco variables son libres; `count_vars` vincula sus coincidencias y la suma, de modo que el solver debe elegir exactamente dos valores 3.

In [29]:
m = cp_model.CpModel()
v = [m.new_int_var(1, 3, f"v_{i}") for i in range(5)]
c = count_vars(m, v, 3, "tres")
m.add(c == 2)
s, estado = resolver(m)
if estado in FACTIBLES:
    valores = [s.value(x) for x in v]
    print(valores)
    assert valores.count(3) == 2

Estado: OPTIMAL
[1, 1, 1, 3, 3]


### Solución 4 — Restricciones adicionales
En cada grupo, la suma de los indicadores de Argentina y Brasil debe ser como máximo 1. En este modelo la regla ya se desprende del límite de Conmebol, por lo que es redundante; sirve para practicar cómo expresar incompatibilidad entre dos equipos.

In [30]:
m_ej, g_ej = crear_grupos(equipos22, federaciones22, bombos22, 8)
ic = equipos22.index("Catar")
ia = equipos22.index("Argentina")
ib = equipos22.index("Brasil")
m_ej.add(g_ej[ic][0] == 1)
for j in range(8):
    m_ej.add(g_ej[ia][j] + g_ej[ib][j] <= 1)
s_ej, estado_ej = resolver(m_ej)
mostrar_y_verificar(s_ej, estado_ej, g_ej, equipos22, federaciones22, bombos22)

Estado: OPTIMAL
Grupo A:
  Catar                  | AFC       | bombo 1
  Uruguay                | Conmebol  | bombo 2
  Senegal                | CAF       | bombo 3
  México                 | Concacaf  | bombo 4
Grupo B:
  Túnez                  | CAF       | bombo 1
  Inglaterra             | UEFA      | bombo 2
  Japón                  | AFC       | bombo 3
  Estados Unidos         | Concacaf  | bombo 4
Grupo C:
  Argentina              | Conmebol  | bombo 1
  Portugal               | UEFA      | bombo 2
  Suiza                  | UEFA      | bombo 3
  Canadá                 | Concacaf  | bombo 4
Grupo D:
  Francia                | UEFA      | bombo 1
  Costa Rica             | Concacaf  | bombo 2
  Serbia                 | UEFA      | bombo 3
  Ecuador                | Conmebol  | bombo 4
Grupo E:
  Países Bajos           | UEFA      | bombo 1
  Bélgica                | UEFA      | bombo 2
  Australia              | AFC       | bombo 3
  Marruecos              | CAF       | bombo 4

### Solución 5 — Palabra inicial
El diccionario traduce una palabra conocida a su identificador entero. Si cambiaste el corpus, se comprueba que la palabra exista antes de añadir la regla.

In [31]:
if "clara" in word_to_i:
    m_ej_texto, seq_ej = crear_texto()
    m_ej_texto.add(seq_ej[0] == word_to_i["clara"])
    s_ej_texto, e_ej_texto = resolver(m_ej_texto)
    if e_ej_texto in FACTIBLES:
        print(formato_poema([s_ej_texto.value(v) for v in seq_ej]))
else:
    print("La palabra 'clara' no está en este corpus.")

Estado: OPTIMAL
clara luna mira suave canción
fría brisa toca tibia pasión
joven nube sueña leve oración
alta estrella busca fresca emoción


### Solución 6 — Capacidad del dominio
Con 15 valores posibles no se pueden asignar 20 valores todos distintos: es el principio del palomar.
Con unicidad por columnas se requieren cuatro valores distintos en cada columna, por lo que 15 palabras no descartan la factibilidad. Las transiciones y las otras restricciones todavía podrían impedir una solución.

## 14. Guía de consulta: del código original a este repaso

| En tus notebooks | En este notebook | Uso |
|---|---|---|
| `CpModel()` | `CpModel()` | Crear modelo |
| `NewIntVar(a,b,n)` | `new_int_var(a,b,n)` | Variable entera |
| `NewBoolVar(n)` | `new_bool_var(n)` | Variable 0/1 |
| `Add(regla)` | `add(regla)` | Añadir restricción |
| `OnlyEnforceIf(b)` | `only_enforce_if(b)` | Activación condicional |
| `b.Not()` | `~b` | Negación booleana |
| `AddMultiplicationEquality` | `add_multiplication_equality` | Producto |
| `AddAllowedAssignments` | `add_allowed_assignments` | Tuplas permitidas |
| `AddForbiddenAssignments` | `add_forbidden_assignments` | Tuplas prohibidas |
| `AddAllDifferent` | `add_all_different` | Distinción global |
| `CpSolver()` | `CpSolver()` | Crear solver |
| `Solve(model)` | `solve(model)` | Resolver |
| `Value(v)` | `value(v)` | Leer decisión |
| `StatusName(status)` | `status_name(status)` | Nombre del estado |
| `CpSolverSolutionCallback` | Igual | Procesar soluciones |
| `enumerate_all_solutions` | Igual | Solicitar enumeración |


## 15. Autoevaluación final
Antes de dar por terminado un modelo, explica con tus palabras:

- Qué decide cada variable y qué representa cada valor de su dominio.
- Qué recorre cada índice y cómo se relaciona con los datos.
- Qué reglas son implicaciones y cuáles son equivalencias.
- Qué datos pueden permanecer como constantes.
- Si buscas una solución, varias soluciones o una óptima.
- Qué estado devolvió el solver y qué puedes concluir de él.
- Cómo verificarás la asignación fuera del modelo.

## Referencias y procedencia

Documentación oficial consultada el 7 de octubre de 2026:
- [CP-SAT: resolución y estados](https://developers.google.com/optimization/cp/cp_solver).
- [Channeling e implicaciones](https://developers.google.com/optimization/cp/channeling).
- [Referencia Python de CpModel](https://or-tools.github.io/docs/python/classortools_1_1sat_1_1python_1_1cp__model_1_1CpModel.html).
- [Código fuente oficial](https://github.com/google/or-tools/blob/stable/ortools/sat/python/cp_model.py).
